# Fine-tuning BiomedBERT on Early MIMIC-III Notes

This notebook performs leakage-controlled contrastive fine-tuning of BiomedBERT on early clinical notes from MIMIC-III. The resulting model, BiomedBERT-MIMIC-Contrastive, is intended for use as a drop-in text embedder in downstream early-risk prediction experiments.

## What this notebook does
- Loads ADMISSIONS.csv and NOTEEVENTS.csv
- Restricts notes to an early hospital window
- Excludes discharge summaries and scrubs explicit outcome-related terms
- Applies patient-level train/validation/test splitting
- Builds training-only contrastive pairs from note chunks
- Fine-tunes a SentenceTransformer model using `MultipleNegativesRankingLoss`
- Saves the resulting fine-tuned encoder to disk

## Data access
This notebook requires authorised access to MIMIC-III via PhysioNet.  
Raw data is not included.

## Expected inputs
Edit the configuration paths in Cell 1 so they point to your local copies of:
- `ADMISSIONS.csv`
- `NOTEEVENTS.csv`

## Output
A saved SentenceTransformer model folder for BiomedBERT-MIMIC-Contrastive, which can then be used by the downstream artifact-generation and evaluation notebooks.

In [ ]:
# =========================
# CELL 1: Paths + Config
# =========================

ADMISSIONS_PATH = "your path to ADMISSIONS.csv"
NOTEEVENTS_PATH = "your path to NOTEEVENTS.csv"

MAX_HOSP_DAY = 2
DEV_SAMPLE = None
SEED = 42

MIN_TEXT_LEN = 20
MAX_NOTES_PER_HADM = 60
CHUNKSIZE = 100_000

# For contrastive fine-tuning pairs (chunking + sampling)
FT_CHUNK_CHARS = 1200
FT_OVERLAP = 200
FT_MAX_CHUNKS_PER_HADM = 80    # cap chunks per admission for balance
PAIRS_PER_HADM = 4             # how many (a,b) pairs per admission

TEST_SIZE = 0.30
VAL_FROM_TEMP = 0.50

In [ ]:
# =========================
# CELL 2: Imports + Reproducibility
# =========================
import re
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import defaultdict

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# =========================
# CELL 3: Helper Functions
# =========================

EXPLICIT_OUTCOME_TERMS = [
    r"\bdeath\b", r"\bdied\b", r"\bexpired\b", r"\bdnr\b", r"\bdni\b",
    r"\bcmo\b", r"\bcomfort measures\b", r"\bcomfort care\b",
    r"\bcardiac arrest\b", r"\barrest\b", r"\bcpr\b", r"\bpost mortem\b",
]
_OUTCOME_PAT = re.compile("|".join(EXPLICIT_OUTCOME_TERMS), flags=re.IGNORECASE)

def scrub_outcome_terms(text_series: pd.Series) -> pd.Series:
    return text_series.astype(str).str.replace(_OUTCOME_PAT, " ", regex=True)

def chunk_text_spread(text: str, chunk_chars: int, overlap: int, max_chunks: int):
    text = str(text)
    if not text.strip():
        return []
    step = max(1, chunk_chars - overlap)
    chunks = [text[i:i + chunk_chars] for i in range(0, len(text), step)]
    if len(chunks) <= max_chunks:
        return chunks
    idx = np.linspace(0, len(chunks) - 1, max_chunks).astype(int)
    return [chunks[i] for i in idx]

def clean_note_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    chunk = chunk.dropna(subset=["HADM_ID", "TEXT"])
    chunk = chunk.query("CATEGORY != 'Discharge summary'")
    chunk["note_date"] = pd.to_datetime(chunk["CHARTDATE"], errors="coerce").dt.normalize()
    chunk = chunk.dropna(subset=["note_date"])
    return chunk

In [ ]:
# =========================
# CELL 4: Load ADMISSIONS (labels)
# =========================
adm = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=["SUBJECT_ID", "HADM_ID", "HOSPITAL_EXPIRE_FLAG", "ADMITTIME"],
    parse_dates=["ADMITTIME"],
)

adm["SUBJECT_ID"] = adm["SUBJECT_ID"].astype(int)
adm["HADM_ID"] = adm["HADM_ID"].astype(int)
adm["label"] = adm["HOSPITAL_EXPIRE_FLAG"].astype(int)
adm["admit_date"] = adm["ADMITTIME"].dt.normalize()

adm = adm[["SUBJECT_ID", "HADM_ID", "admit_date", "label"]]

print("ADMISSIONS shape:", adm.shape)
print(adm["label"].value_counts())

In [ ]:
# =========================
# CELL 5: Stream NOTEEVENTS -> notes_df (filtered + scrubbed)
# =========================
adm_lookup = adm.set_index("HADM_ID")

kept = []
total_rows = 0

for chunk in pd.read_csv(
    NOTEEVENTS_PATH,
    usecols=["HADM_ID", "CATEGORY", "CHARTDATE", "TEXT"],
    chunksize=CHUNKSIZE,
    low_memory=False
):
    total_rows += len(chunk)
    chunk = clean_note_chunk(chunk)
    if chunk.empty:
        continue

    chunk = chunk.join(adm_lookup, on="HADM_ID", how="inner")
    if chunk.empty:
        continue

    chunk["hospital_day"] = (chunk["note_date"] - chunk["admit_date"]).dt.days
    chunk = chunk[(chunk["hospital_day"] >= 0) & (chunk["hospital_day"] <= MAX_HOSP_DAY)]
    if chunk.empty:
        continue

    chunk["TEXT"] = chunk["TEXT"].astype(str)
    chunk = chunk[chunk["TEXT"].str.len() >= MIN_TEXT_LEN]
    if chunk.empty:
        continue

    chunk["text_clean"] = scrub_outcome_terms(chunk["TEXT"])

    kept.append(
        chunk[["SUBJECT_ID", "HADM_ID", "admit_date", "note_date", "hospital_day", "text_clean", "label"]]
    )

notes_df = pd.concat(kept, ignore_index=True)
notes_df["label"] = notes_df["label"].astype(int)

print("Filtered notes (pre-cap):", notes_df.shape)
print(notes_df["label"].value_counts())

In [ ]:
# =========================
# CELL 6: Cap notes per admission
#   - cap within the 0..MAX_HOSP_DAY window
#   - keep deterministic sampling for reproducibility
# =========================
def cap_notes_per_hadm(df: pd.DataFrame, max_notes: int, seed: int = 42) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    out = []
    for hadm_id, g in df.groupby("HADM_ID"):
        if len(g) <= max_notes:
            out.append(g)
        else:
            # sample rows (notes) for that admission
            idx = rng.choice(g.index.values, size=max_notes, replace=False)
            out.append(g.loc[idx])
    return pd.concat(out, ignore_index=False).reset_index(drop=True)

notes_df = cap_notes_per_hadm(notes_df, MAX_NOTES_PER_HADM, seed=SEED)
print("Filtered notes (post-cap):", notes_df.shape)

In [ ]:
# =========================
# CELL 7: Patient-level split (SUBJECT_ID)
#   Produces df_train_notes, df_val_notes, df_test_notes
# =========================
subjects = notes_df["SUBJECT_ID"].unique()
train_subj, temp_subj = train_test_split(subjects, test_size=TEST_SIZE, random_state=SEED)
val_subj, test_subj  = train_test_split(temp_subj, test_size=VAL_FROM_TEMP, random_state=SEED)

df_train_notes = notes_df[notes_df["SUBJECT_ID"].isin(train_subj)].copy()
df_val_notes   = notes_df[notes_df["SUBJECT_ID"].isin(val_subj)].copy()
df_test_notes  = notes_df[notes_df["SUBJECT_ID"].isin(test_subj)].copy()

print("Train/Val/Test notes:", df_train_notes.shape, df_val_notes.shape, df_test_notes.shape)
print("Unique SUBJECT_IDs:", len(train_subj), len(val_subj), len(test_subj))

In [ ]:
# =========================
# CELL 8: Build admission-level "documents" for fine-tuning pairs (TRAIN ONLY)
# Strategy:
#   - For each HADM_ID, concatenate all cleaned notes (within window)
#   - Chunk the concatenated text into FT_CHUNK_CHARS with overlap
#   - Cap chunks per admission to keep balance
# =========================
def build_adm_to_chunks(df_notes: pd.DataFrame,
                        chunk_chars: int,
                        overlap: int,
                        max_chunks_per_adm: int) -> dict:
    adm_to_chunks = {}
    # stable ordering helps reproducibility
    for hadm_id, g in tqdm(df_notes.groupby("HADM_ID"), desc="Building adm_to_chunks"):
        # sort by day then note_date for a stable concatenation
        g = g.sort_values(["hospital_day", "note_date"])
        doc = "\n".join(g["text_clean"].astype(str).tolist())
        chunks = chunk_text_spread(doc, chunk_chars=chunk_chars, overlap=overlap, max_chunks=max_chunks_per_adm)
        if len(chunks) >= 2:
            adm_to_chunks[int(hadm_id)] = chunks
    return adm_to_chunks

adm_to_chunks_train = build_adm_to_chunks(
    df_train_notes,
    chunk_chars=FT_CHUNK_CHARS,
    overlap=FT_OVERLAP,
    max_chunks_per_adm=FT_MAX_CHUNKS_PER_HADM
)

print("Admissions with >=2 chunks (train):", len(adm_to_chunks_train))

In [ ]:
# =========================
# CELL 9: Create (anchor, positive) pairs from same admission (TRAIN ONLY)
# For MultipleNegativesRankingLoss: positives are paired, negatives are in-batch
# =========================
def make_pairs(adm_to_chunks: dict, pairs_per_adm: int, seed: int = 42):
    rng = random.Random(seed)
    pairs = []
    for hadm_id, chunks in adm_to_chunks.items():
        if len(chunks) < 2:
            continue
        for _ in range(pairs_per_adm):
            a, b = rng.sample(chunks, 2)
            pairs.append((a, b))
    rng.shuffle(pairs)
    return pairs

pairs = make_pairs(adm_to_chunks_train, pairs_per_adm=PAIRS_PER_HADM, seed=SEED)
print("Total training pairs:", len(pairs))
print("Example pair lengths:", len(pairs[0][0]), len(pairs[0][1]) if pairs else None)

In [ ]:
# =========================
# CELL 10: Check GPU
# =========================
!nvidia-smi

In [ ]:
# =========================
# CELL 11: Fine-tune BiomedBERT as a SentenceTransformer embedder
#   - Contrastive training with MultipleNegativesRankingLoss
#   - Saves a drop-in model folder you can reuse elsewhere
# =========================
import math
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

# Base encoder (same foundation as your MS-MARCO model, but unfine-tuned for sentence embeddings)
BASE_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

# --- Training knobs ---
MAX_SEQ_LEN = 256          # 128 if you want a quicker/safer first run
BATCH_SIZE = 256           # optimized for 90GB VRAM, be wary of OOM
EPOCHS = 3                 # improvements taper off after 3
LR = 2e-5
WARMUP_FRAC = 0.10
DROP_LAST = True

# If your pairs list is massive, cap it for first run:
# MAX_PAIRS = 300_000
# if len(pairs) > MAX_PAIRS:
#     pairs = pairs[:MAX_PAIRS]
#     print("Capped pairs to:", len(pairs))

# Build InputExamples
train_examples = [InputExample(texts=[a, b]) for (a, b) in pairs]

# Create model
model = SentenceTransformer(BASE_MODEL_NAME)
model.max_seq_length = MAX_SEQ_LEN

# DataLoader
train_loader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    drop_last=DROP_LAST,
    num_workers=2,
    pin_memory=True
)

# Loss
train_loss = losses.MultipleNegativesRankingLoss(model)

# Warmup steps
warmup_steps = math.ceil(len(train_loader) * EPOCHS * WARMUP_FRAC)

print("Examples:", len(train_examples))
print("Batches/epoch:", len(train_loader))
print("Warmup steps:", warmup_steps)
print("Using device:", model.device)

# Train
model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LR},
    show_progress_bar=True
)

In [ ]:
# =========================
# CELL 12: Save the fine-tuned embedder
#   - This path is what to plug into MODEL_PATH in other notebooks
# =========================
SAVE_DIR = "your path to BiomedBERT-MIMIC-Contrastive"
model.save(SAVE_DIR)
print("Saved fine-tuned model to:", SAVE_DIR)

In [ ]:
# =========================
# CELL 13: Quick sanity check
# =========================

m = SentenceTransformer(SAVE_DIR)
emb = m.encode(
    ["Patient stable. No acute distress.", "Patient expired overnight."],
    normalize_embeddings=True
)
print("Embedding shape:", emb.shape)
print("Cosine similarity:", float(np.dot(emb[0], emb[1])))